# Older-to-later model check
This notebook runs our corrected validation check. It uses only the raw Step 3 **training** records. Each fold fits preprocessing on older records, leaves a 31-day gap, and keeps the same animal out of both sides. This is an extra audit, not a new final test result or a replacement for the deployed model.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path.cwd()
assert (repo / 'analysis' / 'temporal_validation.py').is_file(), 'Open this notebook from the PawPath repository folder.'
records = repo.parent / 'step3_results'
assert (records / 'train_records.csv').is_file(), 'Place the Step 3 results beside the PawPath repository.'
subprocess.run([sys.executable, str(repo / 'analysis' / 'temporal_validation.py'), '--records-dir', str(records)], check=True)

## Read the results
Average precision (AP) measures ranking quality across decision cutoffs. Higher is better within the same fold. The long-stay rate changes across folds, so compare the rows carefully. The selected model was **not** the strongest in this audit. See `analysis/EVIDENCE.md` for the viva explanation.

In [ ]:
import pandas as pd
results = pd.read_csv(repo / 'analysis' / 'temporal_validation_results.csv')
display(results[['fold', 'model', 'training_rows', 'validation_rows', 'training_animal_overlap', 'average_precision']])
display(results.groupby('model', as_index=False)['average_precision'].mean().sort_values('average_precision', ascending=False))